In [ ]:
import json
import time
import requests
import pandas as pd
from urllib.parse import urlparse, urljoin

In [5]:
url = "https://ocbc.wd102.myworkdayjobs.com/en-US/External?redirect=/en-US/External/userHome"

In [6]:
response = requests.get(url)
print(response.status_code)

200


In [7]:
parsed_url = urlparse(url)
tenant = parsed_url.netloc.split('.')[0] 
company_name = tenant
print(f"Company name (Tenant): {company_name}")

path_parts = parsed_url.path.strip('/').split('/')
SITE = path_parts[1] if len(path_parts) >= 2 else "External"
BASE = f"https://{parsed_url.netloc}"
API_URL = f"{BASE}/wday/cxs/{tenant}/{SITE}/jobs"
print(f"API URL: {API_URL}")

Company name (Tenant): ocbc
API URL: https://ocbc.wd102.myworkdayjobs.com/wday/cxs/ocbc/External/jobs


In [8]:
listings = []
offset = 0
total = None

PAGE_SIZE = 20

session = requests.Session()
session.headers.update({
    "Content-Type": "application/json",
    "Accept": "application/json",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
})


payload = {
    "appliedFacets": {},
    "limit": PAGE_SIZE,
    "offset": offset,
    "searchText": ""
}
resp = session.post(API_URL, json=payload, timeout=30)
resp.raise_for_status()
data = resp.json()

if total is None:
    total = data.get("total", 0)
    print(f"Total jobs found: {total}")

job_postings = data.get("jobPostings", [])

print(f"Retrieved {len(job_postings)} job postings.")

Total jobs found: 944
Retrieved 20 job postings.


### Code for a single listing

In [9]:
job_posting = job_postings[0]
title = job_posting.get("title", "")
location = job_posting.get("locationsText", "") or job_posting.get("location", "")
posted_date = job_posting.get("postedOn", "")
external_path = job_posting.get("externalPath", "")  # e.g. /job/Singapore/Title_REQID
job_id = job_posting.get("bulletFields", [None])[0] if job_posting.get("bulletFields") else ""

full_link = f"{BASE}/en-US/{SITE}{external_path}" if external_path else ""

print(f"Title: {title}")
print(f"Location: {location}")
print(f"Posted Date: {posted_date}")
print(f"Job ID: {job_id}")
print(f"Full Link: {full_link}")

Title: Risk Data Analyst
Location: OCBC Singapore
Posted Date: Posted Today
Job ID: JR00008946
Full Link: https://ocbc.wd102.myworkdayjobs.com/en-US/External/job/OCBC-Singapore/Risk-Data-Analyst_JR00008946


In [ ]:
from playwright.sync_api import sync_playwright


def main(link):
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()

    page.goto(link, wait_until="domcontentloaded", timeout=3000)
    page.wait_for_selector('[data-automation-id="jobPostingDescription"]', timeout=1500)

    description = page.locator('[data-automation-id="jobPostingDescription"]').inner_text()
    print(f"Job Description: {description[:200]}...")

    browser.close()



Exception in thread Thread-7 (main):
Traceback (most recent call last):
  File "c:\Users\victor\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "c:\Users\victor\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\victor\AppData\Local\Temp\ipykernel_28968\2816017649.py", line 11, in main
  File "c:\Users\victor\AppData\Local\Programs\Python\Python311\Lib\site-packages\playwright\sync_api\_generated.py", line 9770, in goto
    self._sync(
  File "c:\Users\victor\AppData\Local\Programs\Python\Python311\Lib\site-packages\playwright\_impl\_sync_base.py", line 104, in _sync
    raise Error("Event loop is closed! Is Playwright already stopped?")
playwright._impl._errors.Error: Event loop is closed! Is Playwright already stopped?
